<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Preprocesamiento y Data Augmentation (v2.0)
**Proyecto:** Detección de Deslizamientos mediante IA | **EAFIT** 
**Corrección:** Fix de Strides Negativos en PyTorch.

In [ ]:
import os, sys, h5py, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── CONFIGURACIÓN DE RUTAS ──────────────────────────────────────────────────
DATA_ROOT = Path('/content/landslide4sense')
img_dir = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
img_list = sorted(list(img_dir.glob("*.h5")))
mask_list = sorted(list(mask_dir.glob("*.h5")))

CHANNEL_NAMES = ['SAR VV', 'SAR VH', 'SAR VV/VH', 'B2-Blue', 'B3-Green', 'B4-Red', 
                 'B5-RE1', 'B6-RE2', 'B7-RE3', 'B8-NIR', 'B8A-RE4', 'B11-SWIR1', 'B12-SWIR2', 'DEM']

## 1. Clase Dataset con Fix de Memoria (Contiguous Arrays)
Se añade `.copy()` tras las transformaciones de Numpy para evitar el `ValueError` de strides negativos.

In [ ]:
def zscore_norm(patch):
    mean = np.mean(patch, axis=(0, 1))
    std = np.std(patch, axis=(0, 1)) + 1e-6
    return (patch - mean) / std

class LandslideDataset(Dataset):
    def __init__(self, img_files, mask_files, augment=False):
        self.img_files = img_files
        self.mask_files = mask_files
        self.augment = augment

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        with h5py.File(self.img_files[idx], 'r') as hf: 
            img = hf[list(hf.keys())[0]][()].astype(np.float32)
        with h5py.File(self.mask_files[idx], 'r') as hf: 
            mask = hf[list(hf.keys())[0]][()].astype(np.float32)
        
        img = zscore_norm(img)
        
        if self.augment:
            # Rotación aleatoria
            k = np.random.randint(0, 4)
            img = np.rot90(img, k)
            mask = np.rot90(mask, k)
            
            # Flip horizontal
            if np.random.random() > 0.5:
                img = np.flip(img, axis=1)
                mask = np.flip(mask, axis=1)
            
            # CRÍTICO: .copy() para eliminar strides negativos
            img = img.copy()
            mask = mask.copy()

        # Transformar a formato PyTorch [C, H, W]
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask).long()
        return img, mask

## 2. Validación Visual de la Pipeline
Dashboard para verificar que la aumentación y la normalización funcionan correctamente.

In [ ]:
ds = LandslideDataset(img_list, mask_list, augment=True)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
plt.rcParams.update({'font.size': 10, 'font.family': 'sans-serif'})

for i in range(4):
    # Intentamos cargar una muestra que sabemos que tiene datos
    img, mask = ds[min(1816, len(ds)-1)] 
    
    # Re-normalización local para visualización RGB [B5, B4, B3]
    rgb = img[[5, 4, 3]].numpy().transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    
    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"Augmentación Aleatoria {i+1}", fontweight='bold')
    
    # Máscara con mapa de colores térmico
    axes[1, i].imshow(mask, cmap='hot')
    axes[1, i].set_title("Máscara (GT)", color='darkred')
    
    for ax in axes[:, i]: ax.axis('off')

plt.suptitle("Pipeline de Preprocesamiento y Augmentation Geométrica", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("✓ Test finalizado sin errores de memoria.")